# K-Nearest Neighbors for CDC Diabetes Health Indicators

This notebook was created by Liam Russell and uses a previous KNN project of his for inspiration. AI Statement: This code was manually created and checked by Liam Russell and utilized ChatGPT -6.2 Sol for assistance and debugging. The target is `Diabetes_012` with classes **0 = no diabetes**, **1 = prediabetes**, and **2 = diabetes**.

Ensure that `diabetes_012_health_indicators_BRFSS2015.csv` is in the directory beside the notebook if you are going to execute the cells.

In [12]:
# Imports for the KNN.
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [13]:
# Path for the dataset.
csv_path = Path("diabetes_012_health_indicators_BRFSS2015.csv")
df = pd.read_csv(csv_path)
X = df.drop(columns=["Diabetes_012"])
y = df["Diabetes_012"]

# Match the data analysis/engineering notebook's stratified 70/15/15 split.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Training: {len(X_train):,} rows")
print(f"Validation: {len(X_val):,} rows")
print(f"Testing: {len(X_test):,} rows")

Training: 177,576 rows
Validation: 38,052 rows
Testing: 38,052 rows


## Prepare the Features for Euclidean Distance KNN

Create the scaler and fit it on the training split. Standardize BMI, health-day counts, and ordinal variables. Scaling and standardizing are used to prevent the original ranges of some featrues from dominating Euclidean distance. Keep binary indicators at 0 and 1. The label conversion gives `np.bincount` integer class IDs.

In [14]:
# Define the three classes:
class_ids = np.array([0, 1, 2])
class_names = ["No diabetes", "Prediabetes", "Diabetes"]

# Standardize the following features (fit to 0 and 1):
scale_columns = [
    "BMI", "MentHlth", "PhysHlth", "GenHlth", "Age", "Education", "Income"
]
scale_indices = X_train.columns.get_indexer(scale_columns)

started = perf_counter()    # Used for keeping track of the startup time.
scaler = StandardScaler()   # Create teh scaler.
# Store the standard deviation and mean for each training column.
scaler.fit(X_train[scale_columns])  # Fit converts the values using the mean and stdev.

# Builds the array for the KNN (used ChatGPT to help with the feature transformation):
def transform_features(frame):
    """Return a numeric feature array with selected columns standardized."""
    features = frame.loc[:, X_train.columns].to_numpy(dtype=np.float64, copy=True)
    features[:, scale_indices] = scaler.transform(frame[scale_columns])
    return np.ascontiguousarray(features)

# Prepare the labels.
# X_train_knn contains the 177,567 training rows and 21 features:
X_train_knn = transform_features(X_train)   
y_train_knn = np.asarray(y_train, dtype=np.int64)   # One int class label per row.
# Keep track of the startup time.
fit_seconds = perf_counter() - started

## Find Neighbors and Count Votes

Compute Euclidean distances, select the closest training rows, and vote using their labels. This function ranks by squared distance, which gives the same ordering as Euclidean distance. This is to make the function less computationally heavy, as Euclidean distance requires a square root for every distance calculation. This function computes distances in batches and sorts just the selected neighbors. Equal computed distances favor the earlier training rows. If classes receive the same number of votes, the smaller class ID wins.

The class probabilities are the fractions of neighbors voting for each class. For example, votes of `[4, 1, 0]` for five neighbors give probabilities `[0.8, 0.2, 0.0]`.

In [15]:
# Function used to find and order the neighbors.
# Note: x_query are the set of rows to classify. Output rows orders
# the training labels from nearest to farthest. 
def find_neighbor_labels(X_train, y_train, X_query, max_k, batch_size=32):
    """Return the labels of the max_k nearest rows for each query."""
    if not 1 <= max_k <= len(X_train):
        raise ValueError("max_k must be between 1 and the number of training rows.")

    # Creates an empty labels array for the output.
    labels = np.empty((len(X_query), max_k), dtype=y_train.dtype)

    # Computes the distances in batches. ChatGPT helped me find
    # the einsum function and the formatting for this.
    train_squared_norms = np.einsum("ij,ij->i", X_train, X_train)
    # Note: "ij,ij->i" means multiply matching elements and some over j.
    # Rows = i; Columns (Features) = j
    # This gives one squared norm per training row.

    # For each batch, compute the distance using the squared norms.
    for start in range(0, len(X_query), batch_size):
        # Creates the batch.
        batch = X_query[start:start + batch_size]

        # Squared distance = ||query||² + ||training row||² - 2 * dot product.
        # Computes the dot product between the query and training:
        distances = batch @ X_train.T

        distances *= -2.0
        distances += train_squared_norms

        # Squared Norm fo the query row:
        distances += np.einsum("ij,ij->i", batch, batch)[:, None]
        # Clamps small negative values to zero (recommended by ChatGPT).
        np.maximum(distances, 0.0, out=distances)

        # Find the kth distance without sorting all training rows. 
        # Sort only the nearest rows.
        cutoffs = np.partition(distances, max_k - 1, axis=1)[:, max_k - 1].copy()
        for offset, row in enumerate(distances):
            cutoff = cutoffs[offset]    
            closer = np.flatnonzero(row < cutoff)   # Returns index closer than the cutoff.
            tied = np.flatnonzero(row == cutoff)[:max_k - len(closer)]  # Returns the other indecies.
            candidates = np.concatenate((closer, tied))     # Concatenates the closer and tied indecies.

            # Sort by distance first, then by training row index for ties.
            order = np.lexsort((candidates, row[candidates]))
            labels[start + offset] = y_train[candidates[order]]

    return labels

# This function turns the neighbor labels into votes. 
def predict_knn(neighbor_labels, k):
    """Return predicted classes and per-class vote fractions."""
    # Creates the votes array from the labels array.
    # Arrray Format [no diabetes, prediabetes, diabetes]
    votes = np.array([
        np.bincount(labels[:k], minlength=len(class_ids))
        for labels in neighbor_labels
    ])

    # Note: argmax is used is used to handle ties (takes the smaller ID).
    # Note: votes / k finds the vote fractions (ex: 2 votes / 3 classes).
    return class_ids[votes.argmax(axis=1)], votes / k

# This function is used to measure the performance of the predicitons.
def score_predictions(y_true, predictions, probabilities):
    """Compute the same classification metrics for validation and test."""

    # Fraction of the rows where predicted meets the true class.
    accuracy = accuracy_score(y_true, predictions)

    # Returns the accuracy, classification error (wrong rows), log loss and F1 score.
    # Note: the F1 Score and Log Loss are built into sklearn.
    return {
        "Accuracy": accuracy,
        "Classification error": 1 - accuracy,
        "Log loss": log_loss(y_true, probabilities, labels=class_ids),
        "Macro F1": f1_score(
            y_true, predictions, labels=class_ids, average="macro", zero_division=0
        ),
    }

## Select k on Validation

The three candidate ks use the same neighbor search. Evalaute multiple k values (note all are odd). One important note: I originally used just 5, 15, and 31. However, the prediabetes recall was extremely low, so I added more k values. Choose the highest validation macro F1 because prediabetes makes up about 1.83% of the data and accuracy is dominated by the no-diabetes class. The search evaluates every training row per query.

In [16]:
 # Choosen k values to test.
candidate_ks = (1, 3, 5, 7, 9, 11, 15, 31)

started = perf_counter()    # Keep track of the start time.
# Scales the validation column (same method as training):
X_val_knn = transform_features(X_val)   

# Creates an array of the validation neighbor labels (size if the max ks).
val_neighbors = find_neighbor_labels(
    X_train_knn, y_train_knn, X_val_knn, max_k=max(candidate_ks)
)

# Predict and score each of the candidate k values.
validation_rows = []
for k in candidate_ks:
    # Returns the winning class and vote fractions.
    val_predictions, val_probabilities = predict_knn(val_neighbors, k)

    # score_predictions compares the outputs between the predictions and validation set.
    # Keeps track of the k value and scores.
    validation_rows.append({
        "k": k,
        **score_predictions(y_val, val_predictions, val_probabilities),
    })

# Calculates the time for the validation. 
validation_seconds = perf_counter() - started

# Prints the validation results (used ChatGPT to help with printing the results).
validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values(["Macro F1", "k"], ascending=[False, True])
    .reset_index(drop=True)
)
best_k = int(validation_results.loc[0, "k"])
display(validation_results.round(4))
print(f"Selected k: {best_k}")
print(f"Total validation tuning time: {validation_seconds:.2f} s")

,k,Accuracy,Classification error,Log loss,Macro F1
0,3,0.8232,0.1768,3.1327,0.4037
1,5,0.8328,0.1672,2.1485,0.3981
2,1,0.7774,0.2226,8.0239,0.3980
3,7,0.8377,0.1623,1.7434,0.3945
4,9,0.8405,0.1595,1.4832,0.3929
5,11,0.8422,0.1578,1.3132,0.3920
6,15,0.8441,0.1559,1.1083,0.3871
7,31,0.8466,0.1534,0.7783,0.3774


Selected k: 3
Total validation tuning time: 75.18 s


## Evaluate on the Held-Out Test Set

The test scores use the selected value of `k` from the validation. Fit time measures training preprocessing and storage of the training arrays. Test prediction time includes feature transformation, neighbor search, and voting. Perform the testing and print out the results of the KNN.

In [17]:
# Evaluate using the test set (similar structure to the validation and training methods).
started = perf_counter()    # Keep track of the start time.
# Scales the testing column (same method as training):
X_test_knn = transform_features(X_test)

# Finds the nearest training row to each test row (choose the k based on validation).
test_neighbors = find_neighbor_labels(
    X_train_knn, y_train_knn, X_test_knn, max_k=best_k
)
# Counts the closet neighbor's votes and returns their labels.
test_predictions, test_probabilities = predict_knn(test_neighbors, best_k)
# Store the time needed for testing.
prediction_seconds = perf_counter() - started

# Print the results of the KNN. I Used ChatGPT to help create the print statements
# and tables.
knn_results = pd.DataFrame([{
    "Model": "KNN",
    "k": best_k,
    **score_predictions(y_test, test_predictions, test_probabilities),
    "Fit time (s)": fit_seconds,
    "Test prediction time (s)": prediction_seconds,
}])
display(knn_results.round(4))

print(classification_report(
    y_test, test_predictions, labels=class_ids, target_names=class_names,
    digits=4, zero_division=0
))
display(pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=class_ids),
    index=pd.Index(class_names, name="Actual"),
    columns=pd.Index(class_names, name="Predicted"),
))

,Model,k,Accuracy,Classification error,Log loss,Macro F1,Fit time (s),Test prediction time (s)
0,KNN,3,0.8225,0.1775,3.1192,0.403,0.0467,70.8646


              precision    recall  f1-score   support

 No diabetes     0.8678    0.9356    0.9004     32055
 Prediabetes     0.0678    0.0058    0.0106       695
    Diabetes     0.3791    0.2456    0.2981      5302

    accuracy                         0.8225     38052
   macro avg     0.4382    0.3956    0.4030     38052
weighted avg     0.7851    0.8225    0.8002     38052



Predicted,No diabetes,Prediabetes,Diabetes
Actual,,,
No diabetes,29990,42,2023
Prediabetes,582,4,109
Diabetes,3987,13,1302


## Evaluate k Equal to 31

After running the initial testing with k = 3 choosen, based on the F1 score, I decided to retest with k = 31. I was concerned due to the extremely low recall for the prediabetes class (0.58%). Additionally, while k = 31 has a lower F1 score than k = 3, it has better (lower) classification error and log loss.

In [18]:
# Evaluate using the test set (similar structure to the validation and training methods).
started = perf_counter()    # Keep track of the start time.
# Scales the testing column (same method as training):
X_test_knn = transform_features(X_test)

# Finds the nearest training row to each test row (choose the k based on validation).
test_neighbors = find_neighbor_labels(
    X_train_knn, y_train_knn, X_test_knn, max_k=31
)
# Counts the closet neighbor's votes and returns their labels.
test_predictions, test_probabilities = predict_knn(test_neighbors, 31)
# Store the time needed for testing.
prediction_seconds = perf_counter() - started

# Print the results of the KNN. I Used ChatGPT to help create the print statements
# and tables.
knn_results = pd.DataFrame([{
    "Model": "KNN",
    "k": 31,
    **score_predictions(y_test, test_predictions, test_probabilities),
    "Fit time (s)": fit_seconds,
    "Test prediction time (s)": prediction_seconds,
}])
display(knn_results.round(4))

print(classification_report(
    y_test, test_predictions, labels=class_ids, target_names=class_names,
    digits=4, zero_division=0
))
display(pd.DataFrame(
    confusion_matrix(y_test, test_predictions, labels=class_ids),
    index=pd.Index(class_names, name="Actual"),
    columns=pd.Index(class_names, name="Predicted"),
))

,Model,k,Accuracy,Classification error,Log loss,Macro F1,Fit time (s),Test prediction time (s)
0,KNN,31,0.8468,0.1532,0.7851,0.3794,0.0467,71.8578


              precision    recall  f1-score   support

 No diabetes     0.8579    0.9821    0.9158     32055
 Prediabetes     0.0000    0.0000    0.0000       695
    Diabetes     0.5461    0.1396    0.2223      5302

    accuracy                         0.8468     38052
   macro avg     0.4680    0.3739    0.3794     38052
weighted avg     0.7988    0.8468    0.8024     38052



Predicted,No diabetes,Prediabetes,Diabetes
Actual,,,
No diabetes,31481,0,574
Prediabetes,654,0,41
Diabetes,4562,0,740
